# Module 5: RAG Pipeline with Azure DocumentDB

**Time**: ~60 min  
**Environment**: Jupyter notebook in VS Code

Before starting, open a PowerShell terminal in this notebook's folder and run `az login`, followed by `../../../1-DocumentDB-Introduction-and-Cluster-Setup/Set-LabEnvironment.ps1`. Restart the notebook kernel and run each cell in order.

This completed reference uses Microsoft Entra ID for Azure DocumentDB and Azure OpenAI. It creates chunk embeddings, retrieves context with available search features, and builds a grounded prompt.

> Full-text search is a gated preview. When it is unavailable, the notebook reports code 115 and continues with vector-only retrieval.

## Step 0: Connect and configure embeddings

The required Python packages are provided on the workshop VM. This cell reads the three values produced by the shared setup script. Both clients authenticate through `AzureCliCredential`; no access keys are required.


In [ ]:
import os

from azure.identity import AzureCliCredential, get_bearer_token_provider
from pymongo import MongoClient
from pymongo.auth_oidc import OIDCCallback, OIDCCallbackContext, OIDCCallbackResult
from pymongo.errors import OperationFailure
from openai import OpenAI


def require_env(name):
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(
            f"{name} is not set. Run ../../../1-DocumentDB-Introduction-and-Cluster-Setup/"
            "Set-LabEnvironment.ps1, then restart the notebook kernel."
        )
    return value


class AzureIdentityTokenCallback(OIDCCallback):
    def __init__(self, credential):
        self.credential = credential

    def fetch(self, context: OIDCCallbackContext) -> OIDCCallbackResult:
        del context
        token = self.credential.get_token("https://ossrdbms-aad.database.windows.net/.default")
        return OIDCCallbackResult(access_token=token.token)


documentdb_connection_uri = require_env("DOCUMENTDB_CONNECTION_URI")
cluster_name = require_env("DOCUMENTDB_CLUSTER_NAME")
azure_openai_endpoint = require_env("AZURE_OPENAI_ENDPOINT")
embedding_model = require_env("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
credential = AzureCliCredential()
client = MongoClient(
    documentdb_connection_uri,
    tls=True,
    retryWrites=False,
    authMechanism="MONGODB-OIDC",
    authMechanismProperties={"OIDC_CALLBACK": AzureIdentityTokenCallback(credential)},
)
db = client["docdbworkshop"]
chunks = db["rag_chunks"]
openai_client = OpenAI(
    base_url=f"{azure_openai_endpoint.rstrip('/')}/openai/v1/",
    api_key=get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default"),
)
print(db.command({"ping": 1}))
print("DocumentDB cluster:", cluster_name)
print("Embedding deployment:", embedding_model)

## Step 1: Create embeddings for RAG chunks

Each chunk is embedded with OpenAI and stored in Azure DocumentDB with its source metadata.

In [ ]:
def create_embedding(text: str) -> list[float]:
    response = openai_client.embeddings.create(model=embedding_model, input=text)
    return response.data[0].embedding

rag_docs = [
    {"_id":"rag-001","sourceId":"search-module","title":"Vector search","chunk":"Azure DocumentDB vector search uses the $search stage with the cosmosSearch operator to retrieve documents by embedding similarity.","url":"module-4-search","tags":["vector","search"]},
    {"_id":"rag-002","sourceId":"search-module","title":"Full-text search","chunk":"Azure DocumentDB full-text search uses createSearchIndexes and the $search text operator to return BM25-ranked keyword matches.","url":"module-4-search","tags":["full-text","bm25"]},
    {"_id":"rag-003","sourceId":"search-module","title":"Hybrid search","chunk":"Hybrid search runs BM25 keyword retrieval and vector retrieval, then combines ranked lists with Reciprocal Rank Fusion.","url":"module-4-search","tags":["hybrid","rrf"]},
    {"_id":"rag-004","sourceId":"rag-module","title":"Grounded generation","chunk":"A RAG pipeline retrieves relevant chunks from Azure DocumentDB and includes them in the model prompt so the answer is grounded in current application data.","url":"module-5-rag","tags":["rag","generation"]}
]

chunks.drop()
for doc in rag_docs:
    doc["embedding"] = create_embedding(doc["chunk"])
chunks.insert_many(rag_docs)
embedding_dimensions = len(rag_docs[0]["embedding"])
print("Loaded chunks:", chunks.count_documents({}))
print("Embedding dimensions:", embedding_dimensions)

## Step 2: Create retrieval indexes

The vector index enables semantic retrieval. The BM25 search index helps exact terms like `$search`, `BM25`, and `cosmosSearch` rank correctly.

In [ ]:
vector_index_result = db.command({
    "createIndexes": "rag_chunks",
    "indexes": [{
        "name": "idx_chunk_embedding_diskann",
        "key": {"embedding": "cosmosSearch"},
        "cosmosSearchOptions": {"kind": "vector-diskann", "dimensions": embedding_dimensions, "similarity": "COS", "maxDegree": 32, "lBuild": 64}
    }]
})

full_text_search_supported = True
try:
    full_text_index_result = db.command({
        "createSearchIndexes": "rag_chunks",
        "indexes": [{
            "name": "idx_chunk_fts",
            "definition": {"mappings": {"dynamic": False, "fields": {"chunk": {"type": "string"}}}}
        }]
    })
except OperationFailure as error:
    if error.code != 115:
        raise
    full_text_search_supported = False
    full_text_index_result = {"ok": 0, "code": error.code, "message": "Full-text search is not enabled; continue with vector retrieval."}

{"vector": vector_index_result, "fullText": full_text_index_result}


## Step 3: Generate a question embedding and retrieve context

The question is embedded at runtime, then used to retrieve the nearest chunks with `cosmosSearch`.

In [ ]:
question = "How does DocumentDB retrieve context for RAG?"
question_vector = create_embedding(question)
vector_context = list(chunks.aggregate([
    {"$search": {"cosmosSearch": {"path": "embedding", "vector": question_vector, "k": 3}}},
    {"$project": {"_id": 1, "title": 1, "chunk": 1, "url": 1, "score": {"$meta": "searchScore"}}}
]))
vector_context

## Step 4: Hybrid retrieval with RRF

Hybrid retrieval combines BM25 and vector candidates so RAG handles both exact terms and paraphrased questions.

In [ ]:
def rrf(lists, k=60, top_n=3):
    docs, scores = {}, {}
    for results in lists:
        for rank, doc in enumerate(results):
            doc_id = str(doc["_id"])
            docs[doc_id] = doc
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
    return [{**docs[doc_id], "rrfScore": score} for doc_id, score in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]]

if full_text_search_supported:
    keyword_context = list(chunks.aggregate([
        {"$search": {"index": "idx_chunk_fts", "text": {"query": question, "path": "chunk"}}},
        {"$limit": 3},
        {"$project": {"_id": 1, "title": 1, "chunk": 1, "url": 1, "score": {"$meta": "searchScore"}}}
    ]))
    hybrid_context = rrf([keyword_context, vector_context])
else:
    keyword_context = []
    hybrid_context = vector_context
    print("Full-text search is unavailable; using vector context for the prompt.")
hybrid_context


## Step 5: Build the grounded prompt

This prompt includes retrieved DocumentDB chunks and tells the chat model to answer only from that context.

In [ ]:
context_block = "\n\n".join(
    f"[{index + 1}] {document['title']}\n{document['chunk']}\nSource: {document['url']}"
    for index, document in enumerate(hybrid_context)
)
grounded_prompt = f"""You are a helpful assistant for an Azure DocumentDB workshop.
Answer the user's question using only the context below.
If the context does not contain the answer, say you do not know based on the provided context.

<context>
{context_block}
</context>

Question: {question}"""
print(grounded_prompt)
